# ACE generation — graphite

Sample the reference spectrum, generate thermal-scattering ENDF with the selected engine, then use NJOY to create ACE files. `sample_000` is the unperturbed reference.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [ ]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [ ]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import parallel_execution
from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.temperature import resolve_leapr_temperature
elif ENGINE == "ncrystal":
    from input_output.ncrystal.core import NCrystalFile
    import tempfile
    import NCrystal as NC

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/input_2.leapr` | NJOY material template. |
| `BASE_NCMAT_FILE` | `tests/fixtures/C_sg194_pyrolytic_graphite.ncmat` | NCrystal graphite template. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_6-C-012g.jeff` | Neutron evaluation for NJOY processing. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Used for NJOY processing, including ACE generation with either engine. |
| `OUTPUT_DIR` | `tara_output/ace_<engine>/advance/` | Run output directory. |

Edit paths only if your checkout or NJOY installation uses different locations.

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_6-C-012g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if not BASE_ENDF_FILE.is_file():
    raise FileNotFoundError(f"Neutron ENDF missing: {BASE_ENDF_FILE}")
if NJOY_EXECUTABLE is None:
    raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

OUTPUT_DIR = PROJECT_ROOT / "tara_output" / f"ace_{ENGINE}" / "advance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Neutron ENDF:", BASE_ENDF_FILE)
print("NJOY:", NJOY_EXECUTABLE)
print("Output directory:", OUTPUT_DIR)

In [ ]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "input_2.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")

elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "C_sg194_pyrolytic_graphite.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the graphite reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `293.6` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"C"` | Species supplying the reference spectrum. |

In [ ]:
TEMPERATURE_K = 293.6
ATOM_LABEL = "C"
print(f"Engine: {ENGINE_LABEL}")
print(f"temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    leapr_template = template

    resolve_leapr_temperature(template, TEMPERATURE_K, warn_on_reuse=False)
    raw_pdos = template.pdos
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    raw_pdos = material.pdos_for_atom(ATOM_LABEL)
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

## 4. Set the sampling and execution controls

**Required:** Set the two perturbation bounds. Set a positive sample count for a stochastic run.

**Required sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `1.0` | Maximum phonon-energy displacement, in meV. |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative PDOS amplitude change, in percent. |
| `N_SAMPLES` | `3` | Number of stochastic spectra. The reference is held separately. |

**Required for ACE:** the neutron evaluation and NJOY executable from Step 2. Outputs are written to `tara_output/ace_<engine>/advance/`.

In [ ]:
MAX_ENERGY_SHIFT_MEV = 1.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3

**Optional sampling and execution settings**

Edit the following values when you need to change the sampling strategy, reproducibility, or worker allocation.

| Parameter | Example value | What it controls |
|---|---|---|
| `GRID_PARTITION_METHOD` | `'automatic'` | Use automatic or manually specified energy regions. |
| `MANUAL_ENERGY_RANGES` | `None` | Energy ranges in eV, needed only for manual partitioning. |
| `DOMAIN_SPLITTING_STRATEGY` | `'fixed'` | Partition strategy: fixed, deterministic, or random. |
| `CORRELATION_STRATEGY` | `'independent'` | Dependence between domain perturbations: independent or random-group. |
| `VARIABLE_COUPLING` | `'independent'` | Coupling of amplitude and energy-shift variables. |
| `SHARED_GRID_STRUCTURE` | `True` | Whether energy and amplitude perturbations share domains. |
| `FREEZE_ORIGIN` | `True` | Keep the zero-energy acoustic origin fixed. |
| `FAST_AMPLITUDE_CONFIG` | `True` | Enable the fast-amplitude configuration in `TaraConfig`. |
| `AMPLITUDE_METHOD` | `'fast'` | Amplitude perturbation method: fast or precise. |
| `SPATIAL_METHOD` | `'diffeomorphic'` | Spectral energy-grid transformation method: diffeomorphic or crossfade. |
| `AUTO_NORMALIZE` | `True` | Normalize the original PDOS before stochastic sampling. |
| `RANDOM_SEED` | `20261008` | Random seed for reproducible sampling. |
| `BATCH_SIZE` | `512` | Internal sampler chunk size. |
| `N_JOBS` | `1` | Worker request shared by ENDF export and ACE conversion. Use `-1` for automatic allocation. |

In [ ]:
# Set the complete sampling configuration, for this run.
GRID_PARTITION_METHOD = "automatic" # Change to "manual" only with ranges below.
MANUAL_ENERGY_RANGES = None         # For manual mode: [(start_eV, end_eV), ...].
DOMAIN_SPLITTING_STRATEGY = "fixed" # fixed / deterministic / random.
CORRELATION_STRATEGY = "independent"# independent / random_group.
VARIABLE_COUPLING = "independent"  # independent / coupled.
SHARED_GRID_STRUCTURE = True       # Share domain divisions for both perturbations.
FREEZE_ORIGIN = True               # Constrain the zero-energy acoustic boundary.
FAST_AMPLITUDE_CONFIG = True       # Explicit TaraConfig field; see method selector below.
AMPLITUDE_METHOD = "fast"          # "fast" or "precise"; fast aids exploratory runs.
SPATIAL_METHOD = "diffeomorphic"  # "diffeomorphic" or "crossfade".
AUTO_NORMALIZE = True              # Normalize the physical reference spectrum.

RANDOM_SEED = 20261008
BATCH_SIZE = 512
N_JOBS = 1

## 5. Sample the graphite phonon spectrum

**Required:** The PDOS loaded in Section 3 and the sampling settings from Section 4. Build the reference, initialize the sampler, and generate the ensemble in the cells below.

**Optional settings:** None at this step. Adjust sampling in Section 4 and rerun from there.

In [ ]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
    grid_partition_method=GRID_PARTITION_METHOD,
    manual_energy_ranges=MANUAL_ENERGY_RANGES,
    domain_splitting_strategy=DOMAIN_SPLITTING_STRATEGY,
    correlation_strategy=CORRELATION_STRATEGY,
    variable_coupling=VARIABLE_COUPLING,
    shared_grid_structure=SHARED_GRID_STRUCTURE,
    freeze_origin=FREEZE_ORIGIN,
    fast_amplitude=FAST_AMPLITUDE_CONFIG,
)

In [ ]:
# Build the reference spectrum from the selected PDOS.
reference_spectrum = PhononSpectra(
    pdos=raw_pdos,
    auto_normalize=AUTO_NORMALIZE
)

In [ ]:
# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum,
    amplitude_method=AMPLITUDE_METHOD,
    spatial_method=SPATIAL_METHOD,
)

In [ ]:
# Generate the stochastic ensemble with the configured sampling batch size.
ensemble = sampler.generate_ensemble(
    batch_size=BATCH_SIZE
)

if ensemble.reference is None:
    raise RuntimeError("TARA did not provide the expected reference spectrum.")
print(f"Ready: one reference plus {len(ensemble)} stochastic spectra")

## 6. Convert spectra to thermal-scattering ENDF files

**NJOY**

**Required:** The ensemble from Section 5, the LEAPR template and NJOY executable from Section 2, and the temperature from Section 3. LEAPR generates one MF7 tape for the reference and each stochastic sample. The output directory is derived from `OUTPUT_DIR`.

**Optional settings:** None at this step. The batch reuses the worker setting from Section 4.

**NCrystal**

**Required:** The ensemble from Section 5 and the extracted NCMAT, selected atom label, and temperature from Section 3. The output directory is derived from `OUTPUT_DIR`.

**Optional ENDF settings:**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ELASTIC_MODE` | `"mixed"` | Elastic scattering treatment used by NCrystal. |
| `INCLUDE_REFERENCE` | `True` | Export the reference as `sample_000`. With `False`, export only the stochastic samples. |

In [ ]:
from processing.core.config import AceGenerationConfig
from processing.pipelines.ace_pipeline import UniversalAcePipeline

if ENGINE == "njoy":
    from processing.orchestrators.njoy_batch import NjoyBatchOrchestrator
elif ENGINE == "ncrystal":
    from processing.orchestrators.ncrystal_batch import NcrystalBatchOrchestrator

In [ ]:
if ENGINE == "ncrystal":
    # Optional ENDF export settings.
    ELASTIC_MODE = "mixed"
    INCLUDE_REFERENCE = True

# Derive the intermediate output location.
endf_dir = OUTPUT_DIR / "endf"

In [ ]:
# Generate validated ENDF tapes using the selected batch processor.
sab_options = dict(
    ensemble=ensemble,
    output_dir=str(endf_dir),
    temperature=TEMPERATURE_K,
)

if ENGINE == "njoy":
    batch = NjoyBatchOrchestrator(
        njoy_exec=str(NJOY_EXECUTABLE),
        n_jobs=N_JOBS,
    )
    sab_options.update(
        base_leapr_file=str(BASE_LEAPR_FILE),
        material_state="solid",
    )
elif ENGINE == "ncrystal":
    batch = NcrystalBatchOrchestrator(n_jobs=N_JOBS)
    sab_options.update(
        base_ncmat_file=str(BASE_NCMAT_FILE),
        atom_label=ATOM_LABEL,
        elastic_mode=ELASTIC_MODE,
        include_reference=INCLUDE_REFERENCE,
    )

sab_tapes = batch.generate_ensemble_sab_tapes(**sab_options)

print(f"Generated {len(sab_tapes)} validated ENDF tapes in {endf_dir}")

## 7. Generate ACE files with NJOY

**Required:** The neutron ENDF and NJOY executable from Section 2, the temperature from Section 3, and the thermal-scattering tapes from Section 6.

1. **Build the configuration.** `AceGenerationConfig.from_endf()` reads the base evaluation and checks that the tapes in `sab_tapes` can share one configuration.
2. **Run NJOY.** `UniversalAcePipeline.run()` uses `sab_tapes` again to supply the files for conversion. It reuses the worker setting from Section 4.

The optional ACE controls are in the next table. When the shared NJOY input is saved, you can replay it by staging the base ENDF as `tape20` and a sample’s thermal-scattering ENDF as `tape70` in a separate NJOY working directory.

**Optional ACE processing and output settings**

Edit these options in the next code cell if needed.

| Parameter | Example value | What it controls |
|---|---|---|
| `ACE_MAX_ENERGY_EV` | `5.0` | Maximum neutron energy for ACE THERMR processing, in eV. |
| `ACE_SUFFIX` | `'.01'` | Suffix used for the ACE thermal-scattering identifier. |
| `ACE_NBIN` | `16` | Number of THERMR angular bins. |
| `ACE_TOL` | `0.001` | NJOY THERMR interpolation tolerance. |
| `SAVE_XSDIR` | `True` | Save ACE index entries separately under `ace/xsdir/`. |
| `SAVE_NJOY_INPUT` | `True` | Save the shared NJOY deck as `ace/njoy_ace.inp`. |

In [ ]:
# Optional ACE processing and output settings.
ACE_MAX_ENERGY_EV = 5.0
ACE_SUFFIX = ".01"
ACE_NBIN = 16
ACE_TOL = 0.001
SAVE_XSDIR = True
SAVE_NJOY_INPUT = True

In [ ]:
# Build an ACE configuration from validated ENDF metadata and the advanced numerical settings.
ace_config = AceGenerationConfig.from_endf(
    base_endf_path=str(BASE_ENDF_FILE),
    tsl_endf_files=sab_tapes,
    temperature=TEMPERATURE_K,
    emax=ACE_MAX_ENERGY_EV,
    suffix=ACE_SUFFIX,
    nbin=ACE_NBIN,
    tol=ACE_TOL,
)
print(f"Base material: ZA={ace_config.base_za}, MAT={ace_config.base_mat}")
print(f"Thermal material: MAT={ace_config.tsl_mat}, NATOM={ace_config.natom}")
print(f"Elastic treatment: ICOH={ace_config.icoh}, IELAS={ace_config.ielas}")

In [ ]:
# Run the NJOY ACE pipeline, keeping ACE tables separate from optional XSDIR entries.
ace_dir = OUTPUT_DIR / "ace"
ace_pipeline = UniversalAcePipeline(
    njoy_exec=str(NJOY_EXECUTABLE),
    n_jobs=N_JOBS,
)

ace_files = ace_pipeline.run(
    sab_tapes=sab_tapes,
    output_dir=str(ace_dir),
    config=ace_config,
    save_xsdir=SAVE_XSDIR,
    save_njoy_input=SAVE_NJOY_INPUT,
)


print(f"Generated {len(ace_files)} ACE libraries under {ace_dir}")

### Reference: ENDF-derived ACE settings

| Setting | Example value | What it controls |
| --- | --- | --- |
| `base_za`, `base_mat` | Base ENDF, MF1/MT451 | Neutron evaluation identifiers. |
| `tsl_mat` | Thermal ENDF, MF7/MT4 | Thermal-scattering material identifier. |
| `natom` | Thermal ENDF, MF7/MT4, B(6) | Number of principal scattering atoms. |
| `icoh`, `ielas`, `elas_mat` | Thermal ENDF, MF7/MT2, when present | Elastic processing settings. |
| `za1`, `za2`, `za3` | Derived from the base evaluation for this example | ACER moderator nuclide identifiers. |

**NJOY**

The sample IDs are carried forward from the ENDF batch. The reference is `sample_000`. The first stochastic sample is always `sample_001`.

**NCrystal**

The sample IDs are carried forward from the ENDF batch. If the reference is included, it is `sample_000`. The first stochastic sample is always `sample_001`.

## 8. Check the output files

**Required:** The ENDF and ACE output mappings from Sections 6 and 7. Check that each ENDF input has a matching, nonempty ACE file. The cell also checks the XSDIR entries and shared NJOY deck when those outputs were requested.

**Optional settings:** None at this step. Select additional outputs in Section 7.

These checks verify file creation. Use the validation notebooks to assess scattering data for transport.

In [ ]:
# Verify S(alpha, beta) ENDF files, distinct ACE libraries, and optional XSDIR entries.
missing = set(sab_tapes) - set(ace_files)
unexpected = set(ace_files) - set(sab_tapes)
if missing or unexpected:
    raise RuntimeError(
        f"Incomplete ACE batch: missing={sorted(missing)}, unexpected={sorted(unexpected)}"
    )

xsdir_dir = ace_dir / "xsdir"

for sample_id, ace_path in sorted(ace_files.items()):
    # Verify the thermal-scattering ENDF file.
    endf_file = Path(sab_tapes[sample_id])
    if not endf_file.is_file() or endf_file.stat().st_size == 0:
        raise RuntimeError(f"Missing or empty S(alpha, beta) ENDF: {sample_id}: {endf_file}")

    # Verify the ACE library.
    ace_file = Path(ace_path)
    if not ace_file.is_file() or ace_file.stat().st_size == 0:
        raise RuntimeError(f"Missing or empty ACE table: {sample_id}: {ace_file}")

    with ace_file.open("rb") as stream:
        if b"filename route" in stream.read(1024).lower():
            raise RuntimeError(f"XSDIR text was incorrectly saved as ACE: {ace_file}")

    print(f"{sample_id}: ENDF {endf_file.name} ({endf_file.stat().st_size:,} bytes)")
    print(f"{'':>12}  ACE {ace_file.name} ({ace_file.stat().st_size:,} bytes)")

    # Verify the optional XSDIR entry.
    if SAVE_XSDIR:
        xsdir_file = xsdir_dir / f"{sample_id}.xsdir"
        if not xsdir_file.is_file() or xsdir_file.stat().st_size == 0:
            raise RuntimeError(f"Missing or empty XSDIR entry: {sample_id}: {xsdir_file}")

        print(
            f"{'':>12}  XSDIR {xsdir_file.relative_to(ace_dir)} "
            f"({xsdir_file.stat().st_size:,} bytes)"
        )

if SAVE_NJOY_INPUT:
    input_deck = ace_dir / "njoy_ace.inp"
    if not input_deck.is_file() or input_deck.stat().st_size == 0:
        raise RuntimeError(f"Missing or empty NJOY input: {input_deck}")
    print(f"Shared NJOY input: {input_deck}")

print(f"\nFinal ENDF directory: {endf_dir.resolve()}")
print(f"Final ACE directory:  {ace_dir.resolve()}")